
# packages


In [1]:
pip install -q openai PyPDF2 numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.1 MB/s eta 0:00:00


In [2]:
pip install -q tqdm

# openai


In [ ]:
# ===============================
# STEP 2 — OPENAI CLIENT SETUP
# ===============================

# 1️⃣ Import library
from openai import OpenAI
import os

# -------------------------------------------------
# OPTION A (RECOMMENDED in Colab): Use environment variable
# -------------------------------------------------

# Uncomment and paste your key once:
#os.environ["OPENAI_API_KEY"] = 

# -------------------------------------------------
# OPTION B (Manual key — simpler but less secure)
# -------------------------------------------------

# client = OpenAI(api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxx")


# 2️⃣ Test connection (small test call)
try:
    test_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "Say 'OpenAI connection successful'."}
        ],
        temperature=0
    )
    print(test_response.choices[0].message.content)
    print("✅ OpenAI client initialized correctly.")
except Exception as e:
    print("❌ Error connecting to OpenAI:")
    print(e)


# 3️⃣ Embedding helper function (we will use this later in RAG)

def get_embedding(text: str):
    """
    Returns embedding vector for input text.
    Uses text-embedding-3-small (cheap + good for RAG).
    """
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


print("✅ Embedding function ready.")


OpenAI connection successful.
✅ OpenAI client initialized correctly.
✅ Embedding function ready.


Load dataset (pdf)


In [5]:
import PyPDF2

PDF_PATH = "/content/cars_dataset_enlarged.pdf"   # change if needed

def load_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            extracted = page.extract_text() or ""
            text += extracted + "\n"
    return text

raw_text = load_pdf(PDF_PATH)
print(raw_text[:1200])


Brand:
 
Mercedes-Benz
 
 
Model:
 
C-Class
 
C220d
 
Avantgarde
 
 
Price:
 
€28,900
 
 
Year:
 
2019
 
 
Type
 
of
 
car:
 
Saloon
 
 
Mileage:
 
74,800
 
km
 
 
Fuel:
 
Diesel
 
 
Transmission:
 
Automatic
 
 
Colour:
 
Obsidian
 
Black
 
Description:
 
 
The
 
Mercedes-Benz
 
C220d
 
Avantgarde
 
is
 
a
 
refined
 
executive
 
saloon
 
designed
 
for
 
comfort-focused
 
drivers
 
who
 
spend
 
significant
 
time
 
on
 
the
 
road.
 
Its
 
diesel
 
engine
 
delivers
 
strong
 
torque
 
and
 
excellent
 
motorway
 
efficiency,
 
making
 
it
 
particularly
 
suitable
 
for
 
longer
 
commutes
 
or
 
business
 
travel.
 
The
 
automatic
 
transmission
 
ensures
 
smooth
 
and
 
relaxed
 
driving
 
in
 
traffic,
 
while
 
the
 
Avantgarde
 
trim
 
typically
 
emphasizes
 
elegance
 
and
 
comfort
 
over
 
sportiness.
 
Inside,
 
the
 
cabin
 
offers
 
a
 
premium
 
atmosphere
 
with
 
high-quality
 
materials
 
and
 
a
 
quiet
 
ride.
 
This
 
model
 
appeals
 
to
 
professionals
 
seek

Split pdf (brand)

In [6]:
import re

def split_into_car_docs(text: str):
    text = text.replace("\r\n", "\n")
    # Find all positions where a line starts with "Brand:"
    matches = list(re.finditer(r"(?m)^Brand:\s*", text))
    if not matches:
        # fallback: maybe PDF removed line breaks; try a weaker split
        return [t.strip() for t in text.split("Brand:") if t.strip()]

    docs = []
    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        doc = text[start:end].strip()
        if doc:
            docs.append(doc)
    return docs

car_docs = split_into_car_docs(raw_text)
len(car_docs), car_docs[0][:1000]


(140,
 'Brand:\n \nMercedes-Benz\n \n \nModel:\n \nC-Class\n \nC220d\n \nAvantgarde\n \n \nPrice:\n \n€28,900\n \n \nYear:\n \n2019\n \n \nType\n \nof\n \ncar:\n \nSaloon\n \n \nMileage:\n \n74,800\n \nkm\n \n \nFuel:\n \nDiesel\n \n \nTransmission:\n \nAutomatic\n \n \nColour:\n \nObsidian\n \nBlack\n \nDescription:\n \n \nThe\n \nMercedes-Benz\n \nC220d\n \nAvantgarde\n \nis\n \na\n \nrefined\n \nexecutive\n \nsaloon\n \ndesigned\n \nfor\n \ncomfort-focused\n \ndrivers\n \nwho\n \nspend\n \nsignificant\n \ntime\n \non\n \nthe\n \nroad.\n \nIts\n \ndiesel\n \nengine\n \ndelivers\n \nstrong\n \ntorque\n \nand\n \nexcellent\n \nmotorway\n \nefficiency,\n \nmaking\n \nit\n \nparticularly\n \nsuitable\n \nfor\n \nlonger\n \ncommutes\n \nor\n \nbusiness\n \ntravel.\n \nThe\n \nautomatic\n \ntransmission\n \nensures\n \nsmooth\n \nand\n \nrelaxed\n \ndriving\n \nin\n \ntraffic,\n \nwhile\n \nthe\n \nAvantgarde\n \ntrim\n \ntypically\n \nemphasizes\n \nelegance\n \nand\n \ncomfort\n \nover\n

embeddings

In [8]:
def get_embedding(text):
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return resp.data[0].embedding

car_embeddings = [get_embedding(doc) for doc in car_docs]
len(car_embeddings), len(car_embeddings[0])


(140, 1536)

Retrieval

In [9]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def retrieve_top_k(query, docs, embeddings, top_k=10, return_scores=True):
    q_emb = get_embedding(query)
    scores = [cosine_similarity(q_emb, emb) for emb in embeddings]
    top_idx = np.argsort(scores)[::-1][:top_k]
    if return_scores:
        return [(docs[i], float(scores[i])) for i in top_idx]
    else:
        return [docs[i] for i in top_idx]

def build_retrieval_query(user_msg: str):
    return f"""
User request: {user_msg}
Use ALL available fields: Brand, Model, Price, Year, Type of car, Mileage, Fuel, Transmission, Colour,
plus Description, Ideal for, Key advantages, Market segment.
Prioritize: budget, fuel, transmission, body type, and lifestyle needs (family, city driving, long-distance, efficiency).
""".strip()


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a); b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve_top_k(query, docs, embeddings, top_k=6):
    q_emb = get_embedding(query)
    scores = [cosine_similarity(q_emb, embeddings[i]) for i in range(len(docs))]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(docs[i], scores[i]) for i in top_idx]

def build_retrieval_query(user_msg: str):
    return f"""
User request: {user_msg}
Consider: body type, budget, fuel type, transmission, year, mileage,
and lifestyle needs (family, city driving, long-distance, efficiency).
""".strip()



Quick test for retrieval:

pure semantic retrieval test

In [10]:
query = "family hybrid SUV for city driving in Spain"

retrieved = retrieve_top_k(
    build_retrieval_query(query),
    car_docs,
    car_embeddings,
    top_k=8
)

print("\n=== TEST 1: Semantic Retrieval ===\n")
for d, s in retrieved:
    print("SCORE:", round(s, 3))
    print(d.split("\n")[0:6])  # print first 6 lines (metadata)
    print("----")



=== TEST 1: Semantic Retrieval ===

SCORE: 0.691
['Brand:', ' ', 'Hyundai', ' ', ' ', 'Model:']
----
SCORE: 0.675
['Brand:', ' ', 'Honda', ' ', ' ', 'Model:']
----
SCORE: 0.669
['Brand:', ' ', 'MG', ' ', ' ', 'Model:']
----
SCORE: 0.665
['Brand:', ' ', 'Toyota', ' ', ' ', 'Model:']
----
SCORE: 0.664
['Brand:', ' ', 'Suzuki', ' ', ' ', 'Model:']
----
SCORE: 0.663
['Brand:', ' ', 'Honda', ' ', ' ', 'Model:']
----
SCORE: 0.662
['Brand:', ' ', 'Toyota', ' ', ' ', 'Model:']
----
SCORE: 0.66
['Brand:', ' ', 'Toyota', ' ', ' ', 'Model:']
----



creation of constraints to test retrieval


In [11]:
#creation of constraints to test retrieval
import re

def parse_price_eur(doc: str):
    m = re.search(r"Price:\s*€\s*([\d,\.]+)", doc)
    if not m:
        return None
    return int(m.group(1).replace(",", "").replace(".", ""))

def parse_year(doc: str):
    m = re.search(r"Year:\s*(\d{4})", doc)
    return int(m.group(1)) if m else None

def parse_mileage_km(doc: str):
    m = re.search(r"Mileage:\s*([\d,\.]+)\s*km", doc, re.IGNORECASE)
    if not m:
        return None
    return int(m.group(1).replace(",", "").replace(".", ""))

def apply_constraints(results, max_price=None, min_year=None, max_mileage=None):
    filtered = []
    for doc, score in results:
        price = parse_price_eur(doc)
        year = parse_year(doc)
        km = parse_mileage_km(doc)

        if max_price is not None and price is not None and price > max_price:
            continue
        if min_year is not None and year is not None and year < min_year:
            continue
        if max_mileage is not None and km is not None and km > max_mileage:
            continue

        filtered.append((doc, score))

    return filtered


budget constraint filtering test

In [12]:
query = "automatic hybrid SUV"
results = retrieve_top_k(
    build_retrieval_query(query),
    car_docs,
    car_embeddings,
    top_k=15
)

filtered = apply_constraints(results, max_price=30000)

print("\n=== TEST 2: Budget Filtering (<= 30000€) ===\n")

for d, s in filtered[:6]:
    price = parse_price_eur(d)
    print("SCORE:", round(s, 3), "| PRICE:", price)
    print(d.split("\n")[0:6])
    print("----")



=== TEST 2: Budget Filtering (<= 30000€) ===

SCORE: 0.64 | PRICE: 22300
['Brand:', ' ', 'Suzuki', ' ', ' ', 'Model:']
----
SCORE: 0.636 | PRICE: 29900
['Brand:', ' ', 'Renault', ' ', ' ', 'Model:']
----
SCORE: 0.625 | PRICE: 28600
['Brand:', ' ', 'Hyundai', ' ', ' ', 'Model:']
----
SCORE: 0.625 | PRICE: 28900
['Brand:', ' ', 'Honda', ' ', ' ', 'Model:']
----
SCORE: 0.62 | PRICE: 24900
['Brand:', ' ', 'Toyota', ' ', ' ', 'Model:']
----
SCORE: 0.62 | PRICE: 27900
['Brand:', ' ', 'Kia', ' ', ' ', 'Model:']
----


full constraint scenario

In [13]:
query = "automatic hybrid SUV under 30000 euros less than 70000 km"

results = retrieve_top_k(
    build_retrieval_query(query),
    car_docs,
    car_embeddings,
    top_k=20
)

filtered = apply_constraints(
    results,
    max_price=30000,
    max_mileage=70000
)

print("\n=== TEST 3: Full Constraint Filtering ===\n")

for d, s in filtered[:6]:
    price = parse_price_eur(d)
    km = parse_mileage_km(d)
    print("SCORE:", round(s, 3), "| PRICE:", price, "| KM:", km)
    print(d.split("\n")[0:6])
    print("----")



=== TEST 3: Full Constraint Filtering ===

SCORE: 0.671 | PRICE: 29900 | KM: 21600
['Brand:', ' ', 'Renault', ' ', ' ', 'Model:']
----
SCORE: 0.662 | PRICE: 22300 | KM: 38600
['Brand:', ' ', 'Suzuki', ' ', ' ', 'Model:']
----
SCORE: 0.647 | PRICE: 29800 | KM: 49100
['Brand:', ' ', 'Peugeot', ' ', ' ', 'Model:']
----
SCORE: 0.639 | PRICE: 27400 | KM: 18700
['Brand:', ' ', 'Honda', ' ', ' ', 'Model:']
----
SCORE: 0.636 | PRICE: 24900 | KM: 27900
['Brand:', ' ', 'Toyota', ' ', ' ', 'Model:']
----
SCORE: 0.634 | PRICE: 24900 | KM: 55400
['Brand:', ' ', 'MG', ' ', ' ', 'Model:']
----


 recommendations using retrieved context only

In [14]:
def recommend_from_context(user_message, retrieved_docs):
    context = "\n\n---\n\n".join([doc for doc, _ in retrieved_docs])

    messages = [
        {
            "role": "system",
            "content": (
                "You are a car recommendation assistant. "
                "You MUST use ONLY the car dossiers in CONTEXT. "
                "If the user request is missing key constraints (budget, fuel, transmission, body type), "
                "ask 1-2 short follow-up questions before recommending."
            ),
        },
        {
            "role": "user",
            "content": f"""USER_PREFERENCES:
{user_message}

CONTEXT (car dossiers):
{context}

TASK:
Recommend the best 3 cars from CONTEXT.
For each car output exactly:
Brand, Model, Price, Year, Type of car, Mileage, Fuel, Transmission, Colour.
Add 1 short justification sentence per car.
If no good match exists in CONTEXT, say so and ask 1-2 follow-up questions.
""",
        },
    ]

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.3,
    )
    return resp.choices[0].message.content


test if function from step 7 is working (reccommend_from_context)

In [15]:
test_query = "I want an automatic hybrid SUV under 30000 euros which i can take to the beach and to the field"

retrieved = retrieve_top_k(test_query, car_docs, car_embeddings, top_k=6)

response = recommend_from_context(test_query, retrieved)

print(response)


Here are the best 3 cars that match your preferences:

1. **Brand:** Suzuki  
   **Model:** Vitara 1.5 Hybrid GLX  
   **Price:** €22,300  
   **Year:** 2022  
   **Type of car:** Compact SUV  
   **Mileage:** 38,600 km  
   **Fuel:** Hybrid (Gasoline + Electric)  
   **Transmission:** Automatic  
   **Colour:** Blue  
   **Justification:** The Suzuki Vitara offers a compact design with hybrid efficiency, making it a great choice for both beach trips and field adventures.

2. **Brand:** Toyota  
   **Model:** Yaris Cross 1.5 Hybrid Adventure  
   **Price:** €24,900  
   **Year:** 2022  
   **Type of car:** Compact SUV  
   **Mileage:** 27,900 km  
   **Fuel:** Hybrid (Gasoline + Electric)  
   **Transmission:** Automatic  
   **Colour:** Green  
   **Justification:** The Toyota Yaris Cross combines urban efficiency with versatility, perfect for your lifestyle needs.

3. **Brand:** Kia  
   **Model:** Sportage 1.6 Hybrid Drive  
   **Price:** €27,900  
   **Year:** 2021  
   **Type of c

Chatbot

In [16]:
def rag_chat():
    print("Car RAG Chatbot (type 'exit' to stop)\n")
    while True:
        user = input("You: ").strip()
        if user.lower() in ["exit", "quit"]:
            break
        retrieved = retrieve_top_k(user, car_docs, car_embeddings, top_k=8)
        answer = recommend_from_context(user, retrieved)
        print("\nAssistant:\n", answer, "\n")

rag_chat()


Car RAG Chatbot (type 'exit' to stop)

You: i want to buy a 4x4 vehicle for me to drive off-road. make sure it has enough power in the motor 

Assistant:
 Here are the best 3 cars for your off-road driving needs:

1. **Brand:** Jeep  
   **Model:** Wrangler Rubicon  
   **Price:** €48,900  
   **Year:** 2019  
   **Type of car:** Off-road 4x4  
   **Mileage:** 64,000 km  
   **Fuel:** Gasoline  
   **Transmission:** Automatic  
   **Colour:** Black  
   **Justification:** The Jeep Wrangler Rubicon is specifically designed for off-road capability, making it an excellent choice for adventurous driving scenarios.

2. **Brand:** Land Rover  
   **Model:** Defender 110  
   **Price:** €58,000  
   **Year:** 2022  
   **Type of car:** Off-road SUV  
   **Mileage:** 27,000 km  
   **Fuel:** Diesel  
   **Transmission:** Automatic  
   **Colour:** Sand  
   **Justification:** The Land Rover Defender 110 combines rugged off-road capability with modern practicality, ideal for both challenging te

# Explanation of the process / documentation


In [ ]:
#The system follows a classic RAG pipeline

#User Query
#   ↓
#Embedding of Query
#    ↓
#Vector Similarity Search (Retrieval)
#    ↓
#Top-k Relevant Car Dossiers
#    ↓
#LLM Generation Constrained to Context
#    ↓
#Final Car Recommendations